# 02 — Statistical analysis

Targets: **RETURNED** (binary, 11.599%) and `delivery_status` (4-class). Companion: `scripts/eda/stats_analysis.py`.

- §1 univariate moments · §2 categorical tests (χ² + Cramér's V) · numeric gaps (Cohen's d, Mann-Whitney) · focused z-tests on the EDA cliffs · §3 PCA.
- Rule of the house: with n=1M everything is “significant” — **effect sizes decide**, p-values merely confirm.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

CSV = Path("../data/amazon-e-commerce/amazon_ecommerce_1M.csv")
df = pd.read_csv(CSV)
df["is_returned"] = (df["delivery_status"] == "Returned").astype(int)
y = df["is_returned"].to_numpy()
print(f"rows: {len(df):,}  base rate: {y.mean()*100:.3f}%")

NUM = ["price", "discount", "final_price", "rating", "review_count", "stock",
       "seller_rating", "shipping_time_days"]

## §1 · Univariate

In [ ]:
desc = df[NUM].describe(percentiles=[0.25, 0.5, 0.75]).T
desc["skew"] = df[NUM].skew(numeric_only=True)
with pd.option_context("display.float_format", "{:,.2f}".format, "display.width", 200):
    display(desc[["count", "mean", "std", "min", "25%", "50%", "75%", "max", "skew"]])
print("Read: price/final_price heavily right-skewed; stock & seller_rating ~uniform; review_count extreme tail.")

## §2 · Categorical vs RETURNED — χ² finds everything, Cramér's V judges

In [ ]:
import math

def cramers_v(chi2, n, r, k):
    return math.sqrt(chi2 / (n * min(r - 1, k - 1))) if min(r, k) > 1 else 0.0

for c in ["category", "subcategory", "brand", "location", "device", "payment_method", "shipping_time_days"]:
    ct = pd.crosstab(df[c], df["is_returned"])
    chi2, p, dof, _ = stats.chi2_contingency(ct.values)
    v = cramers_v(chi2, len(df), *ct.shape)
    print(f"{c}: V={v:.4f} {'← independent' if v < 0.02 else ''}")
print("\nOnly shipping_time_days clears negligibility (V≈0.09, weak). Everything else: independent.")

## §2 · Numeric gaps — Cohen's d + Mann-Whitney

In [ ]:
for c in NUM:
    a = df.loc[y == 1, c].to_numpy(dtype=float)
    b = df.loc[y == 0, c].to_numpy(dtype=float)
    d = (a.mean() - b.mean()) / math.sqrt((a.var() + b.var()) / 2)
    _, pu = stats.mannwhitneyu(a, b, alternative="two-sided")
    print(f"{c}: d={d:+.4f}  (MWU p={'<1e-300' if pu == 0 else f'{pu:.1g}'})")
print("\n|d| < 0.03 everywhere except rating (−0.13) and ship days (+0.16): small effects at best.")

## §2 · Focused — the two cliffs, with Wilson 95% CIs

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def wilson(p, n, z=1.96):
    d = 1 + z*z/n; c = p + z*z/(2*n); m = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c-m)/d, (c+m)/d

for name, m1, m2 in [("ship==6d vs rest", df["shipping_time_days"] == 6, df["shipping_time_days"] != 6),
                       ("rating<3 vs >=3", df["rating"] < 3.0, df["rating"] >= 3.0)]:
    p1, n1, p2, n2 = y[m1].mean(), m1.sum(), y[m2].mean(), m2.sum()
    z = (p1-p2) / math.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
    print(f"{name}: {p1*100:.2f}% (n={n1:,}) vs {p2*100:.2f}% (n={n2:,})  z={z:.1f} → both cliffs confirmed")

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
g = df.groupby("shipping_time_days")["is_returned"].mean()
ax[0].bar(g.index.astype(str), g.values); ax[0].set_title("return rate by ship days"); ax[0].set_ylim(0, 0.25)
rb = df.assign(rband=pd.cut(df["rating"], [0, 3, 4, 5.1])).groupby("rband", observed=True)["is_returned"].mean()
ax[1].bar([str(i) for i in rb.index], rb.values); ax[1].set_title("return rate by rating band"); ax[1].set_ylim(0, 0.4)
plt.tight_layout(); plt.savefig("/tmp/opencode/stats_cliffs.png"); print("plot: /tmp/opencode/stats_cliffs.png")

## §3 · PCA — what varies together? Does the label separate?

In [ ]:
X = (df[NUM].to_numpy(dtype=float) - df[NUM].mean().to_numpy()) / df[NUM].std().to_numpy()
vals, vecs = np.linalg.eigh(np.cov(X, rowvar=False))
order = np.argsort(vals)[::-1]; vals, vecs = vals[order], vecs[:, order]
for i, v in enumerate(vals):
    print(f"PC{i+1}: {v/vals.sum()*100:5.2f}%  (cum {vals[:i+1].sum()/vals.sum()*100:5.2f}%)")
print(pd.DataFrame(vecs[:, :2], index=NUM, columns=["PC1", "PC2"]).round(3).to_string())
z = X @ vecs[:, :2]
print(f"class-centroid distance in PC1–PC2: {np.linalg.norm(z[y==1].mean(0) - z[y==0].mean(0)):.4f}σ → no practical separation")

In [ ]:
rng = np.random.default_rng(42)
s = rng.choice(len(df), size=20_000, replace=False)
plt.figure(figsize=(5, 4))
plt.scatter(z[s, 0], z[s, 1], c=y[s], s=3, alpha=0.25, cmap="coolwarm")
plt.xlabel("PC1 (money axis)"); plt.ylabel("PC2"); plt.title("PC1 vs PC2, 20k sample (red=returned)")
plt.tight_layout(); plt.savefig("/tmp/opencode/stats_pca.png"); print("plot: /tmp/opencode/stats_pca.png")

## Reading

- **PC1 (28%) is the money axis** (price ⊕ final_price); **PC8 ≈ 0.13% is the money formula itself** — PCA rediscovered `final_price ≈ price·(1−d)` as a near-null dimension. Elegant proof the tolerance story is real.
- **PC2–PC6 sit at ~12.5% each** — the flat spectrum of independent uniform noise.
- **Classes overlap completely** in PC space (0.13σ centroid gap).
- Verdict: global propensity is unlearnable; the two cliffs (§2 focused) are the only exploitable structure — as **rules**, not scores.
- Next: `03_modeling.ipynb` — three algorithm families confirm it the hard way.